# Notebook 02: Data Cleaning, Audit, & Feature Engineering Pipeline
## Nexora Climate Intelligence | CodeFest Datathon Finals 2026

### Purpose & Objectives
This notebook provides complete transparency into the data cleaning, quality auditing, and feature engineering transformations:
1. **Automated Audit Pipeline:** Execute the canonical data audit script and produce `data/outputs/data_quality_audit.csv`.
2. **Country Clean Table:** Merge `co2_emissions_yearly.csv` and `energy_mix_yearly.csv` into `country_clean.csv` (1,350 rows, 0 nulls).
3. **Zero-Leakage Price Features:** Chronologically sort carbon prices and engineer calendar + strictly shifted autoregressive lags and rolling volatility.
4. **Standardize Events & Temperature:** Standardize dates and binary flags for climate events and regional temperature anomalies.
5. **Canonical Data Verification:** Confirm that `data/processed/` contains all 4 single-source-of-truth datasets.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(BASE_DIR))

from src.data_loader import audit_and_clean_all
print('Project root:', BASE_DIR.resolve())
print('Loaded canonical data loader from src/data_loader.py')


---
## 1. Execute Automated Data Quality Audit
Running the automated data cleaning pipeline to generate the canonical datasets and the audit matrix:


In [ ]:
audit_df, clean_tables = audit_and_clean_all(BASE_DIR)

print('=== Canonical Data Quality Audit Report ===')
display(audit_df[['dataset_name', 'raw_rows', 'primary_key', 'missing_cells', 'anomalies_detected', 'clean_rows', 'status']])


---
## 2. Detailed Inspection: `country_clean.csv` (Emissions + Energy Mix)
Merged on `(iso3, year)` to ensure synchronized feature engineering for Question 1.2 and Question 3.
Derived features:
- `clean_baseload_pct = nuclear_pct + hydro_pct`
- `fossil_ratio = fossil_total_pct / (renewables_total_pct + 0.01)`
- `coal_to_gas_ratio = coal_pct / (gas_pct + 0.01)`


In [ ]:
country_clean = clean_tables['country_clean']
print(f'country_clean dimensions: {country_clean.shape[0]} rows x {country_clean.shape[1]} columns')
print('Missing values count:', country_clean.isnull().sum().sum())

# Inspect derived features
display(country_clean[['year', 'country', 'iso3', 'co2_per_capita_t', 'clean_baseload_pct', 'fossil_ratio', 'coal_to_gas_ratio']].head(8))


---
## 3. Detailed Inspection: `prices_clean.csv` & Zero-Leakage Verification
Verifying that rolling windows and lags are strictly shifted backward by at least 1 day:


In [ ]:
prices_clean = clean_tables['prices_clean']
print(f'prices_clean dimensions: {prices_clean.shape[0]} rows x {prices_clean.shape[1]} columns')
print('Missing values count:', prices_clean.isnull().sum().sum())

# Zero-leakage demonstration
eu_sample = prices_clean[prices_clean['market'] == 'EU_ETS'][['date', 'price', 'lag_1', 'lag_2', 'roll_mean_7d', 'roll_std_30d']].tail(8)
display(eu_sample)
print('\nZero Leakage Notice: roll_mean_7d on date T uses prices up to date T-1. No look-ahead bias!')


---
## 4. Detailed Inspection: `events_clean.csv` & `temp_clean.csv`
Standardized climate event flags and monthly temperature anomalies.


In [ ]:
events_clean = clean_tables['events_clean']
temp_clean = clean_tables['temp_clean']

print(f'events_clean: {events_clean.shape[0]} rows x {events_clean.shape[1]} columns')
print(f'temp_clean: {temp_clean.shape[0]} rows x {temp_clean.shape[1]} columns')

display(events_clean.head(5))
display(temp_clean.head(5))


---
## 5. Before vs. After Data Comparison Visualizations
Visualizing fuel share correlations against per-capita emissions across all 50 countries:


In [ ]:
# Correlation matrix of energy mix fuel shares against CO2 per capita
fuel_corr_cols = ['co2_per_capita_t', 'coal_pct', 'oil_pct', 'gas_pct', 'nuclear_pct', 'hydro_pct', 'solar_pct', 'wind_pct', 'clean_baseload_pct', 'fossil_ratio']
corr = country_clean[fuel_corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Fuel Share & Baseload Correlation with CO2 Per Capita', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


---
## 6. Canonical Data Hand-off Summary
All four canonical clean datasets are verified and saved in `data/processed/`:
- `data/processed/country_clean.csv` (1,350 rows x 22 columns)
- `data/processed/prices_clean.csv` (15,866 rows x 23 columns)
- `data/processed/events_clean.csv` (50 rows x 11 columns)
- `data/processed/temp_clean.csv` (2,528 rows x 7 columns)

These files form the single source of truth for Question 1, Question 2, and Question 3.
